In [0]:
class extraccion:

    def consulta_csv(nombre_archivo):
        """
        Funcion para leer un archivo csv.
        Args:
            nombre_archivo (str): nombre del archivo csv.
        Returns:
            df (DataFrame): DataFrame con con los datos del archivo csv.
        """
        try:
            df = spark.read.format("csv").options(
            header=True,
            inferSchema=True
            ).load(f"/Workspace/Users/martin902503@gmail.com/retail_nova/data/{nombre_archivo}.csv")
        except Exception as e:
            print(f"Error funcion: consulta_csv, no se pudo leer el archivo {nombre_archivo}.csv \n Error: {e}")
        return df
    
    def consulta_delta(tabla, capa):
        """
        Funcion para leer una tabla delta.
        Args:
            tabla (str): nombre de la tabla delta.
            capa (str): capa donde se encuentra la tabla delta.
        Returns:
            df (DataFrame): DataFrame con con los datos de la tabla delta.
        """
        try:
            df = spark.sql(f"SELECT * FROM Workspace.{capa}.{tabla}")
        except Exception as e:
            print(f"Error funcion: consulta_delta, no se pudo leer la tabla delta: {tabla} \n Error: {e}")
        return df
    
    

In [0]:
class carga:

    def carga_delta(id_proceso, df, tabla, capa, partitionBy=None):
        """
        Funcion para cargar un DataFrame en delta.
        Args:
            df (DataFrame): DataFrame a cargar.
            tabla (str): nombre de la tabla delta a cargar.
            capa (str): capa donde se va a cargar la tabla delta.
            partitionBy (str): columna por la que se quiere particionar el DataFrame.
        """
        try:
            writer = df.write.mode("overwrite").format("delta")
            if partitionBy:
                writer = writer.partitionBy(partitionBy)
            writer.option("overwriteSchema", "true").saveAsTable(f"Workspace.{capa}.{tabla}")
            configuracion.actualizacion_proceso(capa, id_proceso)
            print(f"Info: La tabla {tabla}, se ha cargado en delta correctamente")
        except Exception as e:
            print(f"Error funcion: carga_delta, no se pudo cargar la tabla {tabla} en delta: \n Error: {e}")

    

In [0]:
class configuracion:

    def configuracion_proceso(capa, id):
        try:
            if id == "":
                df = spark.sql("SELECT * FROM workspace.configuration.configuracion_metadata WHERE capa = '{}'".format(capa))
            else:
                df = spark.sql("SELECT * FROM workspace.configuration.configuracion_metadata WHERE id IN ({})".format(id))
        except Exception as e:
            print(f"Error funcion: configuracion_proceso, no se pudo leer la tabla configuracion_metadata \n Error: {e}")
        return df
    
    def actualizacion_proceso(capa, id_proceso):
        try:
            spark.sql(f"UPDATE workspace.configuration.configuracion_metadata SET flag_procesado = true, fecha_procesado = current_timestamp() WHERE id = {id_proceso}")
        except Exception as e:
            print(f"Error funcion: actualizacion_proceso, no se pudo actualizar la tabla configuracion_metadata: {e}")